# Dataset B — Logistic Regression baseline: CV và full-Train refit

Phạm vi: LR-01 trên Train của split_v1, với temporary training feature lock v0.2 và ma trận LR đã preprocess theo từng fold (dataset_b_preprocessing_v2). Notebook này không chia lại dữ liệu, không fit preprocessing lần nữa, không đọc Validation/Test, không tuning và không chạy RF/XGBoost.

Chọn kernel của môi trường có NumPy, SciPy, pandas, scikit-learn và PyYAML. Output CV hiện tại được chạy lại bằng kernel credit-risk sau khi cài SciPy.

LR-01 giữ bảng dự báo ngoài fold trong biến cv_oof_predictions và kết quả tóm tắt trong output notebook. LR-02 bên dưới tạo fitted pipeline và dự báo Validation trong artifacts/dataset_b/lr_baseline_v1/. Notebook chưa chọn threshold và chưa đánh giá Test.

In [1]:
from pathlib import Path
from importlib.metadata import version
import json
import sys
import time
import warnings

import numpy as np
import pandas as pd
import yaml
from scipy.sparse import load_npz
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = next(
    (
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "config" / "dataset_b" / "model_protocol_v1.yaml").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate the Dataset B project root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_b.preprocessing import DatasetBSplitLoader

with (PROJECT_ROOT / "config/dataset_b/model_protocol_v1.yaml").open(encoding="utf-8") as handle:
    protocol = yaml.safe_load(handle)
with (PROJECT_ROOT / "config/dataset_b/feature_set_v1.yaml").open(encoding="utf-8") as handle:
    feature_contract = yaml.safe_load(handle)
with (PROJECT_ROOT / "config/dataset_b/split_v1.yaml").open(encoding="utf-8") as handle:
    split_contract = yaml.safe_load(handle)

cv_root = PROJECT_ROOT / protocol["contracts"]["cv_artifact"]
with (cv_root / "manifest.json").open(encoding="utf-8") as handle:
    preprocessing_manifest = json.load(handle)

assert protocol["dataset"] == "dataset_b"
assert protocol["protocol_version"] == "model_protocol_v1"
assert protocol["contracts"]["preprocessing_version"] == "dataset_b_preprocessing_v2"
assert feature_contract["feature_version"] == preprocessing_manifest["feature_version"] == "temporary_training_lock_v0_2"
assert len(feature_contract["model_input_raw_columns"]) == 12
assert split_contract["split_version"] == preprocessing_manifest["split_version"] == "split_v1"
assert preprocessing_manifest["preprocessing_version"] == "dataset_b_preprocessing_v2"
assert preprocessing_manifest["source_partition"] == "train"
assert preprocessing_manifest["fit_scope"] == "fold_train_only"
assert preprocessing_manifest["use"] == "cross_validation_only"
assert preprocessing_manifest["encoded_feature_count"] == 66

expected_cv = {
    "splitter": "StratifiedKFold",
    "n_splits": 5,
    "shuffle": True,
    "random_state": 42,
}
for key, expected in expected_cv.items():
    assert protocol["cross_validation"][key] == expected
    assert split_contract["cross_validation"][key] == expected
assert protocol["cross_validation"]["population"] == "train_only"
assert split_contract["cross_validation"]["allowed_population"] == "train_only"
assert protocol["cross_validation"]["hyperparameter_search"] is False
assert protocol["cross_validation"]["trial_budget_per_model"] == 1
assert protocol["cross_validation"]["primary_metric"] == "roc_auc"
assert len(preprocessing_manifest["folds"]) == 5
assert [fold["fold"] for fold in preprocessing_manifest["folds"]] == [1, 2, 3, 4, 5]

lr_contract = protocol["models"]["lr"]
assert lr_contract["estimator"] == "sklearn.linear_model.LogisticRegression"
assert lr_contract["preprocessing_profile"] == "lr"
assert lr_contract["parameters"]["class_weight"] is None
assert protocol["class_handling"]["sample_weight"] is False
assert protocol["class_handling"]["resampling"] is False
lr_parameters = dict(lr_contract["parameters"])

train_partition = DatasetBSplitLoader().load("train")
train_ids = train_partition.record_ids.to_numpy()
train_y = train_partition.y.to_numpy()
n_train = len(train_ids)
assert n_train == 205310
assert len(np.unique(train_ids)) == n_train
assert set(np.unique(train_y)) == {0, 1}
print(f"Train rows: {n_train:,}; CV folds: 5; encoded columns: 66")
print(f"LR parameters: {lr_parameters}")

Train rows: 205,310; CV folds: 5; encoded columns: 66
LR parameters: {'penalty': 'l2', 'C': 1.0, 'solver': 'lbfgs', 'max_iter': 1000, 'fit_intercept': True, 'class_weight': None}


In [2]:
oof_probability = np.full(n_train, np.nan, dtype=float)
oof_fold = np.zeros(n_train, dtype=np.int8)
fold_results = []

for fold_contract in preprocessing_manifest["folds"]:
    fold_number = fold_contract["fold"]
    fold_dir = cv_root / fold_contract["directory"]
    X_fold_train = load_npz(fold_dir / "lr_train.npz")
    X_fold_validation = load_npz(fold_dir / "lr_fold_validation.npz")
    with np.load(fold_dir / "rows.npz", allow_pickle=False) as rows:
        fit_positions = rows["train_positions"]
        hold_positions = rows["fold_validation_positions"]
        fit_ids = rows["train_record_ids"]
        hold_ids = rows["fold_validation_record_ids"]
        fit_y = rows["train_y"]
        hold_y = rows["fold_validation_y"]

    assert X_fold_train.shape == (fold_contract["train_rows"], 66)
    assert X_fold_validation.shape == (fold_contract["fold_validation_rows"], 66)
    assert len(fit_positions) == len(fit_ids) == len(fit_y)
    assert len(hold_positions) == len(hold_ids) == len(hold_y)
    assert np.array_equal(train_ids[fit_positions], fit_ids)
    assert np.array_equal(train_ids[hold_positions], hold_ids)
    assert np.array_equal(train_y[fit_positions], fit_y)
    assert np.array_equal(train_y[hold_positions], hold_y)
    assert len(np.unique(np.concatenate([fit_positions, hold_positions]))) == n_train
    assert not np.any(oof_fold[hold_positions])
    assert np.isfinite(X_fold_train.data).all()
    assert np.isfinite(X_fold_validation.data).all()

    estimator = LogisticRegression(**lr_parameters)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        start_fit = time.perf_counter()
        estimator.fit(X_fold_train, fit_y)
        fit_seconds = time.perf_counter() - start_fit

    if not np.array_equal(estimator.classes_, [0, 1]):
        raise ValueError(f"Fold {fold_number}: unexpected class order {estimator.classes_}.")
    start_predict = time.perf_counter()
    probability = estimator.predict_proba(X_fold_validation)[:, 1]
    predict_seconds = time.perf_counter() - start_predict
    if not np.isfinite(probability).all() or np.any((probability < 0) | (probability > 1)):
        raise ValueError(f"Fold {fold_number}: invalid probabilities.")

    auc = float(roc_auc_score(hold_y, probability))
    oof_probability[hold_positions] = probability
    oof_fold[hold_positions] = fold_number
    warning_messages = [f"{item.category.__name__}: {item.message}" for item in caught]
    convergence_warnings = [
        message for message, item in zip(warning_messages, caught)
        if issubclass(item.category, ConvergenceWarning)
    ]
    fold_results.append({
        "fold": fold_number,
        "train_rows": len(fit_y),
        "fold_validation_rows": len(hold_y),
        "roc_auc": auc,
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        "n_iter": int(np.max(estimator.n_iter_)),
        "warnings": warning_messages,
        "convergence_warnings": convergence_warnings,
    })
    print(
        f"Fold {fold_number}: ROC-AUC={auc:.6f}, fit={fit_seconds:.2f}s, "
        f"predict={predict_seconds:.2f}s, warnings={len(warning_messages)}"
    )
    for message in warning_messages:
        print(f"  WARNING: {message}")

Fold 1: ROC-AUC=0.807765, fit=1.68s, predict=0.00s, warnings=1
Fold 2: ROC-AUC=0.805888, fit=1.18s, predict=0.00s, warnings=1
Fold 3: ROC-AUC=0.809927, fit=1.32s, predict=0.00s, warnings=1
Fold 4: ROC-AUC=0.805587, fit=0.97s, predict=0.00s, warnings=1
Fold 5: ROC-AUC=0.810503, fit=1.55s, predict=0.00s, warnings=1


In [3]:
if np.isnan(oof_probability).any():
    raise ValueError("Some Train records have no out-of-fold probability.")
if np.any(oof_fold == 0):
    raise ValueError("Some Train records were not assigned a validation fold.")
if not np.isfinite(oof_probability).all():
    raise ValueError("Out-of-fold probabilities contain non-finite values.")

cv_oof_predictions = pd.DataFrame({
    "record_id": train_ids,
    "y_approved": train_y,
    "fold": oof_fold,
    "p_approved": oof_probability,
})
assert len(cv_oof_predictions) == n_train
assert cv_oof_predictions["record_id"].is_unique
assert cv_oof_predictions["fold"].value_counts().sort_index().to_dict() == {
    fold["fold"]: fold["fold_validation_rows"]
    for fold in preprocessing_manifest["folds"]
}

auc_values = np.asarray([result["roc_auc"] for result in fold_results], dtype=float)
has_warnings = any(result["warnings"] for result in fold_results)
cv_summary = {
    "status": "complete_with_warnings" if has_warnings else "complete",
    "model": "lr",
    "protocol_version": protocol["protocol_version"],
    "feature_version": feature_contract["feature_version"],
    "split_version": split_contract["split_version"],
    "preprocessing_version": preprocessing_manifest["preprocessing_version"],
    "population": "train_only",
    "fold_count": len(fold_results),
    "train_rows": n_train,
    "mean_roc_auc": float(auc_values.mean()),
    "std_roc_auc": float(auc_values.std(ddof=0)),
    "std_ddof": 0,
    "parameters": lr_parameters,
    "library_versions": {
        "numpy": np.__version__,
        "scipy": version("scipy"),
        "scikit-learn": version("scikit-learn"),
    },
    "folds": fold_results,
}
print(f"CV status: {cv_summary['status']}")
print(f"Mean ROC-AUC: {cv_summary['mean_roc_auc']:.6f}")
print(f"Std ROC-AUC (ddof=0): {cv_summary['std_roc_auc']:.6f}")
print(f"Out-of-fold predictions: {len(cv_oof_predictions):,} unique Train records")
display(pd.DataFrame(fold_results).drop(columns=["warnings", "convergence_warnings"]))
display(cv_oof_predictions.head())

CV status: complete_with_warnings
Mean ROC-AUC: 0.807934
Std ROC-AUC (ddof=0): 0.002015
Out-of-fold predictions: 205,310 unique Train records


,fold,train_rows,fold_validation_rows,roc_auc,fit_seconds,predict_seconds,n_iter
0,1,164248,41062,0.807765,1.681477,0.001223,203
1,2,164248,41062,0.805888,1.184336,0.001204,146
2,3,164248,41062,0.809927,1.324610,0.001111,167
3,4,164248,41062,0.805587,0.971990,0.001336,117
4,5,164248,41062,0.810503,1.553156,0.001313,182


,record_id,y_approved,fold,p_approved
0,dataset_b-000343629,1,5,0.931241
1,dataset_b-000256425,1,4,0.913824
2,dataset_b-000342696,1,2,0.820508
3,dataset_b-000235315,0,2,0.626459
4,dataset_b-000271985,1,5,0.920489


## Phạm vi kết quả

cv_summary chứa metric, tham số, version, thời gian và cảnh báo theo fold. cv_oof_predictions chứa dự báo ngoài fold theo record_id trong thứ tự Train gốc. Phần CV không xuất dự báo ngoài fold ra CSV; phần LR-02 bên dưới chỉ lưu fitted pipeline và dự báo Validation. Không dùng CV để chọn threshold hoặc báo cáo Test.

## LR-02 — Fit toàn Train và dự báo Validation

Bước này tạo pipeline mới từ 12 raw features. Preprocessing và LR chỉ fit trên toàn Train; Validation chỉ dùng để lấy P(y_approved=1). Không chọn threshold, không tính metric Validation, không đọc Test. Artifact được ghi trong artifacts/dataset_b/lr_baseline_v1/ và không ghi đè artifact CV.

In [4]:
import platform
import joblib
from sklearn.pipeline import Pipeline
from src.dataset_b.preprocessing import make_preprocessing_pipeline

loader = DatasetBSplitLoader()
full_train = loader.load("train")
validation = loader.load("validation")
expected_features = feature_contract["model_input_raw_columns"]
assert list(full_train.X.columns) == expected_features
assert list(validation.X.columns) == expected_features
assert len(full_train.X) == 205310
assert len(validation.X) == 43995
assert np.array_equal(full_train.record_ids.to_numpy(), train_ids)
assert not np.intersect1d(full_train.record_ids.to_numpy(), validation.record_ids.to_numpy()).size
assert not set(expected_features) & set(feature_contract["audit_only_columns"])
assert not set(expected_features) & {"record_id", "action_taken", "y_approved"}
assert protocol["refit"]["population"] == "full_train"
assert protocol["refit"]["reinitialize_preprocessing"] is True
assert protocol["refit"]["reinitialize_model"] is True
assert protocol["validation"]["use"] == "threshold_selection"
assert protocol["validation"]["may_refit_preprocessing_or_model"] is False

lr_full_pipeline = Pipeline([
    ("preprocess", make_preprocessing_pipeline("lr")),
    ("estimator", LogisticRegression(**lr_parameters)),
])
with warnings.catch_warnings(record=True) as full_fit_warnings:
    warnings.simplefilter("always")
    start_fit = time.perf_counter()
    lr_full_pipeline.fit(full_train.X, full_train.y)
    full_fit_seconds = time.perf_counter() - start_fit

warning_messages = [
    f"{item.category.__name__}: {item.message}"
    for item in full_fit_warnings
]
if any(issubclass(item.category, ConvergenceWarning) for item in full_fit_warnings):
    raise RuntimeError("Full-Train LR emitted a convergence warning; do not publish artifacts.")
assert np.array_equal(lr_full_pipeline.named_steps["estimator"].classes_, [0, 1])
assert lr_full_pipeline.named_steps["preprocess"].transform(validation.X.iloc[:1]).shape[1] == 66

start_predict = time.perf_counter()
validation_probability = lr_full_pipeline.predict_proba(validation.X)[:, 1]
validation_predict_seconds = time.perf_counter() - start_predict
if not np.isfinite(validation_probability).all():
    raise ValueError("Validation probabilities contain non-finite values.")
if np.any((validation_probability < 0) | (validation_probability > 1)):
    raise ValueError("Validation probabilities are outside [0, 1].")

validation_predictions = pd.DataFrame({
    "record_id": validation.record_ids.to_numpy(),
    "y_approved": validation.y.to_numpy(),
    "p_approved": validation_probability,
})
assert len(validation_predictions) == 43995
assert validation_predictions["record_id"].is_unique
print(f"Full-Train LR fitted on {len(full_train.X):,} rows in {full_fit_seconds:.2f}s.")
print(f"Validation probabilities generated for {len(validation_predictions):,} rows in {validation_predict_seconds:.2f}s.")
print(f"Fit warnings: {len(warning_messages)}")
for message in warning_messages:
    print(f"  WARNING: {message}")

Dataset B LTV >1000% retained for 42 rows
Dataset B LTV >1000% retained for 7 rows


Full-Train LR fitted on 205,310 rows in 4.97s.
Validation probabilities generated for 43,995 rows in 0.41s.
Fit warnings: 1


In [5]:
artifact_dir = PROJECT_ROOT / "artifacts" / "dataset_b" / "lr_baseline_v1"
model_path = artifact_dir / "lr_full_train.joblib"
prediction_path = artifact_dir / "validation_predictions.csv"
manifest_path = artifact_dir / "run_manifest.json"
artifact_paths = (model_path, prediction_path, manifest_path)

run_manifest = {
    "schema_version": 1,
    "dataset": "dataset_b",
    "run_id": "lr_baseline_v1",
    "stage": "full_train_refit_validation_predictions",
    "model": "lr",
    "model_protocol_version": protocol["protocol_version"],
    "feature_version": feature_contract["feature_version"],
    "split_version": split_contract["split_version"],
    "preprocessing_version": preprocessing_manifest["preprocessing_version"],
    "fit_partition": "train",
    "fit_rows": len(full_train.X),
    "prediction_partition": "validation",
    "prediction_rows": len(validation_predictions),
    "raw_feature_columns": expected_features,
    "encoded_feature_count": 66,
    "model_parameters": lr_parameters,
    "class_order": [0, 1],
    "probability_column": "p_approved",
    "fit_seconds": full_fit_seconds,
    "validation_predict_seconds": validation_predict_seconds,
    "fit_warnings": warning_messages,
    "library_versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": version("scipy"),
        "scikit-learn": version("scikit-learn"),
        "joblib": joblib.__version__,
    },
    "artifacts": {
        "fitted_pipeline": model_path.name,
        "validation_predictions": prediction_path.name,
        "run_manifest": manifest_path.name,
    },
}

if any(path.exists() for path in artifact_paths):
    if not all(path.is_file() for path in artifact_paths):
        raise FileExistsError("Partial LR run artifacts exist; refusing to overwrite them.")
    with manifest_path.open(encoding="utf-8") as handle:
        existing_manifest = json.load(handle)
    for key in (
        "dataset", "run_id", "model", "model_protocol_version",
        "feature_version", "split_version", "preprocessing_version",
        "fit_partition", "fit_rows", "prediction_partition", "prediction_rows",
        "raw_feature_columns", "encoded_feature_count", "model_parameters",
        "class_order", "probability_column",
    ):
        if existing_manifest.get(key) != run_manifest[key]:
            raise FileExistsError(f"Existing LR artifact has a different {key}; refusing to overwrite.")
    if existing_manifest.get("library_versions") != run_manifest["library_versions"]:
        raise FileExistsError("Existing LR artifact uses different library versions; refusing to overwrite.")
    print("Existing LR artifacts found; verifying without overwriting.")
else:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(lr_full_pipeline, model_path)
    validation_predictions.to_csv(prediction_path, index=False, float_format="%.17g")
    with manifest_path.open("w", encoding="utf-8") as handle:
        json.dump(run_manifest, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
    print("Created LR full-Train model, Validation predictions, and run manifest.")

loaded_pipeline = joblib.load(model_path)
saved_predictions = pd.read_csv(prediction_path, dtype={"record_id": str})
with manifest_path.open(encoding="utf-8") as handle:
    saved_manifest = json.load(handle)
assert list(saved_predictions.columns) == ["record_id", "y_approved", "p_approved"]
assert len(saved_predictions) == len(validation_predictions)
assert saved_predictions["record_id"].is_unique
assert np.array_equal(saved_predictions["record_id"].to_numpy(), validation_predictions["record_id"].to_numpy())
assert np.array_equal(saved_predictions["y_approved"].to_numpy(), validation_predictions["y_approved"].to_numpy())
assert np.allclose(saved_predictions["p_approved"].to_numpy(), validation_probability, rtol=0, atol=1e-15)
assert np.array_equal(loaded_pipeline.named_steps["estimator"].classes_, [0, 1])
reloaded_probability = loaded_pipeline.predict_proba(validation.X)[:, 1]
assert np.allclose(reloaded_probability, validation_probability, rtol=0, atol=1e-12)
assert saved_manifest["feature_version"] == feature_contract["feature_version"]
assert saved_manifest["split_version"] == split_contract["split_version"]
assert saved_manifest["preprocessing_version"] == preprocessing_manifest["preprocessing_version"]
print(f"Reload validation passed: {len(saved_predictions):,} ordered record IDs and matching probabilities.")
display(saved_predictions.head())

Existing LR artifacts found; verifying without overwriting.


Dataset B LTV >1000% retained for 7 rows


Reload validation passed: 43,995 ordered record IDs and matching probabilities.


,record_id,y_approved,p_approved
0,dataset_b-000299096,1,0.836618
1,dataset_b-000172161,0,0.821545
2,dataset_b-000065193,0,0.229796
3,dataset_b-000291201,0,0.610123
4,dataset_b-000137115,1,0.914288


## LR-03 — Chọn và khóa threshold LR trên Validation

Dùng 999 ngưỡng k/1000 với k = 1,...,999; dự báo Approve khi p_approved >= t. Tối đa hóa Balanced Accuracy, xử lý đồng hạng bằng khoảng cách đến 0.5 rồi chọn ngưỡng thấp hơn. Dữ liệu và nhãn đều thuộc Validation; không fit lại model và không đọc Test. Metric tại ngưỡng đã chọn trên chính Validation là kết quả chọn ngưỡng của run experimental feature v0.2, không phải đánh giá RQ1 cuối.


In [6]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

lr03_artifact_dir = PROJECT_ROOT / "artifacts" / "dataset_b" / "lr_baseline_v1"
lr03_prediction_path = lr03_artifact_dir / "validation_predictions.csv"
lr03_manifest_path = lr03_artifact_dir / "run_manifest.json"
lr03_validation_path = PROJECT_ROOT / "data" / "dataset_b" / "split_v1" / "validation.csv"
lr03_candidates_path = lr03_artifact_dir / "threshold_candidates.csv"
lr03_selection_path = lr03_artifact_dir / "threshold_selection.json"

with lr03_manifest_path.open(encoding="utf-8") as handle:
    lr03_manifest = json.load(handle)
assert lr03_manifest["run_id"] == "lr_baseline_v1"
assert lr03_manifest["prediction_partition"] == "validation"
assert lr03_manifest["feature_version"] == "temporary_training_lock_v0_2"
assert lr03_manifest["probability_column"] == "p_approved"

lr03_predictions = pd.read_csv(lr03_prediction_path, dtype={"record_id": str})
lr03_validation = pd.read_csv(
    lr03_validation_path,
    usecols=["record_id", "y_approved"],
    dtype={"record_id": str},
)
assert list(lr03_predictions.columns) == ["record_id", "y_approved", "p_approved"]
assert len(lr03_predictions) == lr03_manifest["prediction_rows"] == len(lr03_validation)
assert lr03_predictions["record_id"].is_unique
assert np.array_equal(
    lr03_predictions["record_id"].to_numpy(),
    lr03_validation["record_id"].to_numpy(),
)
assert np.array_equal(
    lr03_predictions["y_approved"].to_numpy(),
    lr03_validation["y_approved"].to_numpy(),
)

lr03_y = lr03_predictions["y_approved"].to_numpy()
lr03_p = lr03_predictions["p_approved"].to_numpy()
assert set(np.unique(lr03_y)) == {0, 1}
assert np.isfinite(lr03_p).all()
assert np.all((0 <= lr03_p) & (lr03_p <= 1))
lr03_positive = lr03_y == 1
lr03_negative = lr03_y == 0
lr03_n_approved = int(lr03_positive.sum())
lr03_n_denied = int(lr03_negative.sum())
lr03_grid = np.arange(1, 1000, dtype=np.int64)
lr03_thresholds = lr03_grid / 1000.0

lr03_tp = np.array(
    [np.count_nonzero(lr03_p[lr03_positive] >= t) for t in lr03_thresholds],
    dtype=np.int64,
)
lr03_tn = np.array(
    [np.count_nonzero(lr03_p[lr03_negative] < t) for t in lr03_thresholds],
    dtype=np.int64,
)
lr03_fn = lr03_n_approved - lr03_tp
lr03_fp = lr03_n_denied - lr03_tn
# Fixed class denominators make this integer score exactly rank Balanced Accuracy.
lr03_score = lr03_tp * lr03_n_denied + lr03_tn * lr03_n_approved
lr03_max_indices = np.flatnonzero(lr03_score == lr03_score.max())
lr03_selected_index = min(
    lr03_max_indices,
    key=lambda i: (abs(int(lr03_grid[i]) - 500), int(lr03_grid[i])),
)
lr03_balanced_accuracy = lr03_score / (2 * lr03_n_approved * lr03_n_denied)

lr03_candidates = pd.DataFrame({
    "threshold": lr03_thresholds,
    "tp_approved": lr03_tp,
    "fn_approved": lr03_fn,
    "tn_denied": lr03_tn,
    "fp_denied": lr03_fp,
    "n_approved": lr03_n_approved,
    "n_denied": lr03_n_denied,
    "recall_approved": lr03_tp / lr03_n_approved,
    "recall_denied": lr03_tn / lr03_n_denied,
    "balanced_accuracy": lr03_balanced_accuracy,
    "is_maximum": lr03_score == lr03_score.max(),
    "is_selected": np.arange(len(lr03_grid)) == lr03_selected_index,
})
assert len(lr03_candidates) == 999
assert int(lr03_candidates["is_selected"].sum()) == 1

lr03_csv = lr03_candidates.to_csv(index=False, float_format="%.17g")
lr03_selection = {
    "schema_version": 1,
    "run_id": lr03_manifest["run_id"],
    "stage": "validation_threshold_selection",
    "feature_version": lr03_manifest["feature_version"],
    "split_version": lr03_manifest["split_version"],
    "preprocessing_version": lr03_manifest["preprocessing_version"],
    "model_protocol_version": lr03_manifest["model_protocol_version"],
    "prediction_partition": "validation",
    "prediction_file": lr03_prediction_path.name,
    "prediction_sha256": hashlib.sha256(lr03_prediction_path.read_bytes()).hexdigest(),
    "candidate_file": lr03_candidates_path.name,
    "candidate_sha256": hashlib.sha256(lr03_csv.encode("utf-8")).hexdigest(),
    "candidate_grid": {"start": 0.001, "stop": 0.999, "step": 0.001, "inclusive": True, "count": 999},
    "prediction_rule": "p_approved >= threshold -> Approve (y_approved=1)",
    "objective": "maximize_validation_balanced_accuracy",
    "tie_break": ["minimum_absolute_distance_to_0.5", "lower_threshold"],
    "maximizer_count": int(len(lr03_max_indices)),
    "maximizing_thresholds": [int(lr03_grid[i]) / 1000 for i in lr03_max_indices],
    "selected_threshold": int(lr03_grid[lr03_selected_index]) / 1000,
    "selected_metrics": {
        "balanced_accuracy": float(lr03_balanced_accuracy[lr03_selected_index]),
        "recall_approved": float(lr03_tp[lr03_selected_index] / lr03_n_approved),
        "recall_denied": float(lr03_tn[lr03_selected_index] / lr03_n_denied),
    },
    "validation_counts": {
        "rows": int(len(lr03_predictions)),
        "n_approved": lr03_n_approved,
        "n_denied": lr03_n_denied,
        "tp_approved": int(lr03_tp[lr03_selected_index]),
        "fn_approved": int(lr03_fn[lr03_selected_index]),
        "tn_denied": int(lr03_tn[lr03_selected_index]),
        "fp_denied": int(lr03_fp[lr03_selected_index]),
    },
    "evaluation_status": "experimental_feature_v0_2_validation_selection_only",
}
lr03_json = json.dumps(lr03_selection, ensure_ascii=False, indent=2) + "\n"

if lr03_candidates_path.exists() or lr03_selection_path.exists():
    if not (lr03_candidates_path.is_file() and lr03_selection_path.is_file()):
        raise FileExistsError("Partial LR-03 artifacts exist; refusing to overwrite.")
    if lr03_candidates_path.read_bytes() != lr03_csv.encode("utf-8"):
        raise FileExistsError("Existing threshold candidate table differs; refusing to overwrite.")
    if lr03_selection_path.read_text(encoding="utf-8") != lr03_json:
        raise FileExistsError("Existing threshold selection differs; refusing to overwrite.")
    print("Existing LR-03 artifacts match; no overwrite.")
else:
    lr03_candidates_path.write_text(lr03_csv, encoding="utf-8", newline="")
    lr03_selection_path.write_text(lr03_json, encoding="utf-8", newline="")
    print("Created LR-03 threshold artifacts.")

print(
    f"Validation: {len(lr03_predictions):,} rows; "
    f"selected threshold {lr03_selection['selected_threshold']:.3f}; "
    f"Balanced Accuracy {lr03_selection['selected_metrics']['balanced_accuracy']:.6f}; "
    f"{lr03_selection['maximizer_count']} maximizer(s)."
)


Existing LR-03 artifacts match; no overwrite.
Validation: 43,995 rows; selected threshold 0.747; Balanced Accuracy 0.739151; 1 maximizer(s).


## LR-04 — Dự báo Test đúng một lần

Dùng fitted LR pipeline của LR-02 và threshold 0.747 của LR-03. Kiểm tra feature, split, preprocessing, model và hash trước khi dự báo. Tạo một bộ 1.000 stratified bootstrap indices chung ở cấp Dataset B; lưu Test predictions theo record_id. Khi artifact đã có, cell chỉ kiểm tra và dùng lại, không gọi predict_proba lần hai.


In [7]:
from pathlib import Path
import hashlib
import json
import time
import joblib
import numpy as np
import pandas as pd
import yaml
from src.dataset_b.preprocessing import DatasetBSplitLoader

lr04_root = PROJECT_ROOT
lr04_dir = lr04_root / "artifacts" / "dataset_b"
lr04_run_dir = lr04_dir / "lr_baseline_v1"
lr04_model_path = lr04_run_dir / "lr_full_train.joblib"
lr04_manifest_path = lr04_run_dir / "run_manifest.json"
lr04_threshold_path = lr04_run_dir / "threshold_selection.json"
lr04_prediction_path = lr04_run_dir / "test_predictions.csv"
lr04_indices_path = lr04_dir / "rq1_test_bootstrap_indices.npz"
lr04_indices_manifest_path = lr04_dir / "rq1_test_bootstrap_manifest.json"
lr04_split_path = lr04_dir / "split_v1.json"

with lr04_manifest_path.open(encoding="utf-8") as handle:
    lr04_run_manifest = json.load(handle)
with lr04_threshold_path.open(encoding="utf-8") as handle:
    lr04_threshold_selection = json.load(handle)
with (lr04_root / "config/dataset_b/model_protocol_v1.yaml").open(encoding="utf-8") as handle:
    lr04_protocol = yaml.safe_load(handle)
with (lr04_root / "config/dataset_b/feature_set_v1.yaml").open(encoding="utf-8") as handle:
    lr04_feature_contract = yaml.safe_load(handle)

assert lr04_run_manifest["run_id"] == lr04_threshold_selection["run_id"] == "lr_baseline_v1"
assert lr04_run_manifest["feature_version"] == lr04_threshold_selection["feature_version"] == lr04_feature_contract["feature_version"]
assert lr04_run_manifest["split_version"] == lr04_threshold_selection["split_version"] == "split_v1"
assert lr04_run_manifest["preprocessing_version"] == lr04_threshold_selection["preprocessing_version"]
assert lr04_run_manifest["model_protocol_version"] == lr04_threshold_selection["model_protocol_version"] == lr04_protocol["protocol_version"]
assert lr04_run_manifest["model_parameters"] == lr04_protocol["models"]["lr"]["parameters"]
assert lr04_protocol["test"]["may_change_preprocessing_model_or_threshold"] is False
assert lr04_threshold_selection["selected_threshold"] == 0.747
assert hashlib.sha256((lr04_run_dir / "validation_predictions.csv").read_bytes()).hexdigest() == lr04_threshold_selection["prediction_sha256"]
assert hashlib.sha256((lr04_run_dir / "threshold_candidates.csv").read_bytes()).hexdigest() == lr04_threshold_selection["candidate_sha256"]

lr04_test = DatasetBSplitLoader().load("test")
assert len(lr04_test.X) == 43996
assert list(lr04_test.X.columns) == lr04_run_manifest["raw_feature_columns"]
assert len(lr04_test.X.columns) == 12
assert not set(lr04_test.X.columns) & set(lr04_test.audit.columns)
lr04_ids = lr04_test.record_ids.to_numpy()
lr04_y = lr04_test.y.to_numpy(dtype=np.int8)
assert len(set(lr04_ids)) == len(lr04_ids)
assert set(np.unique(lr04_y)) == {0, 1}
lr04_ids_hash = hashlib.sha256(("\n".join(lr04_ids) + "\n").encode("utf-8")).hexdigest()
lr04_model_hash = hashlib.sha256(lr04_model_path.read_bytes()).hexdigest()
lr04_threshold_hash = hashlib.sha256(lr04_threshold_path.read_bytes()).hexdigest()

# A Dataset B level index matrix is shared by LR, RF, and XGBoost.
if lr04_indices_path.exists() or lr04_indices_manifest_path.exists():
    if not (lr04_indices_path.is_file() and lr04_indices_manifest_path.is_file()):
        raise FileExistsError("Partial shared Test bootstrap artifact exists.")
    with lr04_indices_manifest_path.open(encoding="utf-8") as handle:
        lr04_indices_manifest = json.load(handle)
    assert lr04_indices_manifest["test_record_id_sha256"] == lr04_ids_hash
    assert lr04_indices_manifest["split_version"] == "split_v1"
    assert lr04_indices_manifest["replicates"] == 1000
    assert lr04_indices_manifest["seed"] == 42
    assert lr04_indices_manifest["class_counts"] == {"0": int((lr04_y == 0).sum()), "1": int((lr04_y == 1).sum())}
    assert hashlib.sha256(lr04_indices_path.read_bytes()).hexdigest() == lr04_indices_manifest["indices_sha256"]
    print("Reusing shared Dataset B bootstrap indices.")
else:
    lr04_class0 = np.flatnonzero(lr04_y == 0).astype(np.int32)
    lr04_class1 = np.flatnonzero(lr04_y == 1).astype(np.int32)
    lr04_rng = np.random.default_rng(42)
    lr04_indices = np.empty((1000, len(lr04_y)), dtype=np.int32)
    for lr04_rep in range(1000):
        lr04_indices[lr04_rep, :len(lr04_class0)] = lr04_rng.choice(lr04_class0, size=len(lr04_class0), replace=True)
        lr04_indices[lr04_rep, len(lr04_class0):] = lr04_rng.choice(lr04_class1, size=len(lr04_class1), replace=True)
    np.savez_compressed(lr04_indices_path, indices=lr04_indices)
    lr04_indices_manifest = {
        "schema_version": 1,
        "dataset": "dataset_b",
        "split_version": "split_v1",
        "partition": "test",
        "test_rows": len(lr04_y),
        "test_record_id_sha256": lr04_ids_hash,
        "split_manifest_sha256": hashlib.sha256(lr04_split_path.read_bytes()).hexdigest(),
        "indices_file": lr04_indices_path.name,
        "indices_sha256": hashlib.sha256(lr04_indices_path.read_bytes()).hexdigest(),
        "index_base": 0,
        "shape": [1000, len(lr04_y)],
        "dtype": "int32",
        "replicates": 1000,
        "seed": 42,
        "rng": "numpy.random.Generator(PCG64)",
        "numpy_version": np.__version__,
        "sampling": "within each true class, sample row positions with replacement at original class count; concatenate class 0 then class 1",
        "class_counts": {"0": len(lr04_class0), "1": len(lr04_class1)},
        "reuse": ["lr", "rf", "xgboost"],
    }
    lr04_indices_manifest_path.write_text(json.dumps(lr04_indices_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8", newline="")
    del lr04_indices
    print("Saved 1,000 shared stratified Test bootstrap index rows.")

if lr04_prediction_path.exists():
    if "test_inference" not in lr04_run_manifest:
        raise FileExistsError("Test prediction exists without saved inference metadata; refusing a second inference.")
    lr04_predictions = pd.read_csv(lr04_prediction_path, dtype={"record_id": str})
    assert hashlib.sha256(lr04_prediction_path.read_bytes()).hexdigest() == lr04_run_manifest["test_inference"]["prediction_sha256"]
    assert lr04_run_manifest["test_inference"]["model_sha256"] == lr04_model_hash
    assert lr04_run_manifest["test_inference"]["threshold_sha256"] == lr04_threshold_hash
    print("Reusing saved Test predictions; no model inference.")
else:
    if "test_inference" in lr04_run_manifest:
        raise FileExistsError("Inference metadata exists without Test prediction file.")
    lr04_pipeline = joblib.load(lr04_model_path)
    lr04_estimator = lr04_pipeline.named_steps["estimator"]
    assert type(lr04_estimator).__name__ == "LogisticRegression"
    assert np.array_equal(lr04_estimator.classes_, [0, 1])
    assert int(lr04_estimator.n_features_in_) == lr04_run_manifest["encoded_feature_count"] == 66
    assert len(lr04_pipeline.named_steps["preprocess"].get_feature_names_out()) == 66
    for lr04_key, lr04_value in lr04_run_manifest["model_parameters"].items():
        assert lr04_estimator.get_params()[lr04_key] == lr04_value
    lr04_start = time.perf_counter()
    lr04_p = lr04_pipeline.predict_proba(lr04_test.X)[:, 1]
    lr04_inference_seconds = time.perf_counter() - lr04_start
    assert len(lr04_p) == len(lr04_y)
    assert np.isfinite(lr04_p).all() and np.all((lr04_p >= 0) & (lr04_p <= 1))
    lr04_predicted = (lr04_p >= 0.747).astype(np.int8)
    lr04_predictions = pd.DataFrame({
        "record_id": lr04_ids,
        "y_approved": lr04_y,
        "p_approved": lr04_p,
        "p_denied": 1.0 - lr04_p,
        "predicted_approved": lr04_predicted,
        "predicted_denied": 1 - lr04_predicted,
    })
    lr04_predictions.to_csv(lr04_prediction_path, index=False, float_format="%.17g")
    lr04_run_manifest["test_inference"] = {
        "partition": "test",
        "rows": len(lr04_y),
        "model_sha256": lr04_model_hash,
        "threshold_sha256": lr04_threshold_hash,
        "selected_threshold": 0.747,
        "prediction_file": lr04_prediction_path.name,
        "prediction_sha256": hashlib.sha256(lr04_prediction_path.read_bytes()).hexdigest(),
        "bootstrap_indices_file": lr04_indices_path.name,
        "bootstrap_indices_sha256": lr04_indices_manifest["indices_sha256"],
        "inference_seconds_including_preprocessing": lr04_inference_seconds,
        "predict_proba_calls_on_test": 1,
    }
    lr04_manifest_path.write_text(json.dumps(lr04_run_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8", newline="")
    print(f"Saved one Test prediction pass: {len(lr04_y):,} rows in {lr04_inference_seconds:.3f}s.")

assert list(lr04_predictions.columns) == ["record_id", "y_approved", "p_approved", "p_denied", "predicted_approved", "predicted_denied"]
assert np.array_equal(lr04_predictions["record_id"].to_numpy(), lr04_ids)
assert np.array_equal(lr04_predictions["y_approved"].to_numpy(), lr04_y)
assert np.isfinite(lr04_predictions["p_approved"]).all()
assert np.allclose(lr04_predictions["p_denied"], 1.0 - lr04_predictions["p_approved"], rtol=0, atol=1e-15)
assert np.array_equal(lr04_predictions["predicted_approved"], lr04_predictions["p_approved"] >= 0.747)


Reusing shared Dataset B bootstrap indices.
Reusing saved Test predictions; no model inference.


## LR-04 — RQ1 point metrics và 95% bootstrap CI

Tính ROC-AUC trên P(Approved), average precision cho Denied, các metric phân lớp tại threshold cố định, Brier, calibration và thời gian. Calibration curve dùng 10 quantile bins; intercept là logistic offset với slope bằng 1, còn slope từ logistic recalibration có intercept tự do. CI percentile 2.5% và 97.5% từ 1.000 bootstrap replicates cho scalar predictive metrics. Invalid metric được ghi theo replicate và loại khỏi CI của chính metric đó. Curve và thời gian không nhận CI từ bootstrap prediction. Kết quả là RQ1 evaluation của lr_baseline_v1 với experimental feature lock v0.2, không phải FINAL_MODEL_FEATURES của toàn dự án.


In [8]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize, root
from scipy.special import expit, logit
from sklearn.metrics import average_precision_score, roc_auc_score

lr04_run_dir = PROJECT_ROOT / "artifacts" / "dataset_b" / "lr_baseline_v1"
lr04_shared_dir = PROJECT_ROOT / "artifacts" / "dataset_b"
lr04_manifest_path = lr04_run_dir / "run_manifest.json"
lr04_prediction_path = lr04_run_dir / "test_predictions.csv"
lr04_metrics_path = lr04_run_dir / "rq1_test_metrics.json"
lr04_bootstrap_path = lr04_run_dir / "rq1_bootstrap_metrics.csv"
lr04_indices_path = lr04_shared_dir / "rq1_test_bootstrap_indices.npz"
lr04_indices_manifest_path = lr04_shared_dir / "rq1_test_bootstrap_manifest.json"
with lr04_manifest_path.open(encoding="utf-8") as handle:
    lr04_run_manifest = json.load(handle)
with lr04_indices_manifest_path.open(encoding="utf-8") as handle:
    lr04_indices_manifest = json.load(handle)
lr04_prediction_sha = hashlib.sha256(lr04_prediction_path.read_bytes()).hexdigest()
assert lr04_prediction_sha == lr04_run_manifest["test_inference"]["prediction_sha256"]
assert hashlib.sha256(lr04_indices_path.read_bytes()).hexdigest() == lr04_indices_manifest["indices_sha256"]
assert lr04_run_manifest["test_inference"]["bootstrap_indices_sha256"] == lr04_indices_manifest["indices_sha256"]
assert lr04_run_manifest["test_inference"]["predict_proba_calls_on_test"] == 1
lr04_predictions = pd.read_csv(lr04_prediction_path, dtype={"record_id": str})
assert list(lr04_predictions.columns) == ["record_id", "y_approved", "p_approved", "p_denied", "predicted_approved", "predicted_denied"]
assert len(lr04_predictions) == lr04_run_manifest["test_inference"]["rows"] == lr04_indices_manifest["test_rows"] == 43996
assert lr04_predictions["record_id"].is_unique
lr04_y = lr04_predictions["y_approved"].to_numpy(dtype=np.int8)
lr04_p = lr04_predictions["p_approved"].to_numpy(dtype=np.float64)
lr04_threshold = float(lr04_run_manifest["test_inference"]["selected_threshold"])
assert lr04_threshold == 0.747
assert set(np.unique(lr04_y)) == {0, 1}
assert np.isfinite(lr04_p).all() and np.all((lr04_p >= 0) & (lr04_p <= 1))
assert np.allclose(lr04_predictions["p_denied"], 1 - lr04_p, rtol=0, atol=1e-15)
assert np.array_equal(lr04_predictions["predicted_approved"], (lr04_p >= lr04_threshold).astype(np.int8))
assert np.array_equal(lr04_predictions["predicted_denied"], (lr04_p < lr04_threshold).astype(np.int8))
lr04_ids_sha = hashlib.sha256(("\n".join(lr04_predictions["record_id"]) + "\n").encode("utf-8")).hexdigest()
assert lr04_ids_sha == lr04_indices_manifest["test_record_id_sha256"]

lr04_scalar_names = [
    "roc_auc_approved", "average_precision_denied",
    "precision_denied", "recall_denied", "f1_denied",
    "recall_approved", "balanced_accuracy", "brier_approved",
    "calibration_intercept", "calibration_slope",
]

def lr04_calibration(y, p):
    z = logit(np.clip(p, 1e-15, 1 - 1e-15))
    target = y.astype(np.float64)
    invalid = {}
    intercept = np.nan
    slope = np.nan
    try:
        intercept = float(brentq(
            lambda a: float(np.mean(expit(z + a) - target)),
            -80, 80, xtol=1e-10,
        ))
    except (ArithmeticError, ValueError, RuntimeError) as exc:
        invalid["calibration_intercept"] = str(exc)

    def loss_and_gradient(theta):
        eta = theta[0] + theta[1] * z
        error = expit(eta) - target
        loss = float(np.mean(np.logaddexp(0, eta) - target * eta))
        gradient = np.array([np.mean(error), np.mean(error * z)], dtype=np.float64)
        return loss, gradient

    try:
        slope_fit = minimize(
            loss_and_gradient,
            x0=np.array([0.0, 1.0]),
            method="L-BFGS-B",
            jac=True,
            options={"maxiter": 100, "ftol": 1e-12, "gtol": 1e-9},
        )
        slope_parameters = slope_fit.x
        if not slope_fit.success or not np.isfinite(slope_parameters).all():
            initial = slope_parameters if np.isfinite(slope_parameters).all() else np.array([0.0, 1.0])
            fallback = root(lambda theta: loss_and_gradient(theta)[1], initial, method="hybr", tol=1e-10)
            slope_parameters = fallback.x
        score = loss_and_gradient(slope_parameters)[1]
        if not np.isfinite(slope_parameters).all() or not np.isfinite(score).all() or np.max(np.abs(score)) > 1e-7:
            raise ArithmeticError("Calibration slope score did not converge.")
        slope = float(slope_parameters[1])
    except (ArithmeticError, ValueError, RuntimeError) as exc:
        invalid["calibration_slope"] = str(exc)
    return intercept, slope, invalid

def lr04_metrics(y, p, threshold):
    invalid = {}
    values = {}
    predicted_approved = p >= threshold
    approved = y == 1
    denied = y == 0
    tp = int(np.count_nonzero(approved & predicted_approved))
    fn = int(np.count_nonzero(approved & ~predicted_approved))
    tn = int(np.count_nonzero(denied & ~predicted_approved))
    fp = int(np.count_nonzero(denied & predicted_approved))
    n_approved = tp + fn
    n_denied = tn + fp
    if n_approved == 0 or n_denied == 0:
        raise ArithmeticError("Both true classes are required.")
    values["roc_auc_approved"] = float(roc_auc_score(y, p))
    values["average_precision_denied"] = float(average_precision_score(1 - y, 1 - p))
    if tn + fn:
        values["precision_denied"] = tn / (tn + fn)
    else:
        values["precision_denied"] = np.nan
        invalid["precision_denied"] = "no_predicted_denied"
    values["recall_denied"] = tn / n_denied
    values["f1_denied"] = 2 * tn / (2 * tn + fn + fp)
    values["recall_approved"] = tp / n_approved
    values["balanced_accuracy"] = (values["recall_approved"] + values["recall_denied"]) / 2
    values["brier_approved"] = float(np.mean((p - y) ** 2))
    values["calibration_intercept"], values["calibration_slope"], calibration_invalid = lr04_calibration(y, p)
    invalid.update(calibration_invalid)
    for name, value in values.items():
        if not np.isfinite(value):
            invalid.setdefault(name, "nonfinite_metric")
    return values, invalid, {
        "tp_approved": tp, "fn_approved": fn,
        "tn_denied": tn, "fp_denied": fp,
        "n_approved": n_approved, "n_denied": n_denied,
    }

def lr04_curve(y, p):
    edges = np.unique(np.quantile(p, np.linspace(0, 1, 11)))
    if len(edges) < 2:
        raise ArithmeticError("Calibration curve has fewer than two distinct edges.")
    bin_id = np.searchsorted(edges[1:-1], p, side="right")
    bins = []
    for i in range(len(edges) - 1):
        selected = bin_id == i
        if not selected.any():
            continue
        bins.append({
            "bin": i,
            "lower_edge": float(edges[i]),
            "upper_edge": float(edges[i + 1]),
            "count": int(selected.sum()),
            "mean_p_approved": float(p[selected].mean()),
            "observed_approval_rate": float(y[selected].mean()),
        })
    assert sum(item["count"] for item in bins) == len(y)
    return bins

if lr04_metrics_path.exists() or lr04_bootstrap_path.exists():
    if not (lr04_metrics_path.is_file() and lr04_bootstrap_path.is_file()):
        raise FileExistsError("Partial LR-04 metric artifacts exist.")
    evaluation = lr04_run_manifest.get("rq1_test_evaluation")
    if evaluation is None:
        raise FileExistsError("Metric artifacts exist without run manifest references.")
    assert hashlib.sha256(lr04_metrics_path.read_bytes()).hexdigest() == evaluation["metrics_sha256"]
    assert hashlib.sha256(lr04_bootstrap_path.read_bytes()).hexdigest() == evaluation["bootstrap_metrics_sha256"]
    print("Existing LR-04 metrics match; no recomputation.")
else:
    point, point_invalid, counts = lr04_metrics(lr04_y, lr04_p, lr04_threshold)
    if point_invalid:
        raise ArithmeticError(f"Test point metrics are invalid: {point_invalid}")
    curve = lr04_curve(lr04_y, lr04_p)
    with np.load(lr04_indices_path, allow_pickle=False) as package:
        indices = package["indices"]
    assert indices.shape == (1000, len(lr04_y))
    assert indices.dtype == np.int32
    assert np.all((indices >= 0) & (indices < len(lr04_y)))
    n_denied = counts["n_denied"]
    assert np.all(lr04_y[indices[:, :n_denied]] == 0)
    assert np.all(lr04_y[indices[:, n_denied:]] == 1)
    rows = []
    for replicate_id, row_indices in enumerate(indices):
        sampled, invalid, _ = lr04_metrics(lr04_y[row_indices], lr04_p[row_indices], lr04_threshold)
        rows.append({
            "replicate_id": replicate_id,
            **sampled,
            "invalid_metrics": json.dumps(invalid, ensure_ascii=False, sort_keys=True),
        })
        if (replicate_id + 1) % 100 == 0:
            print(f"Bootstrap {replicate_id + 1}/1000", flush=True)
    bootstrap_frame = pd.DataFrame(rows, columns=["replicate_id", *lr04_scalar_names, "invalid_metrics"])
    assert len(bootstrap_frame) == 1000
    ci = {}
    for name in lr04_scalar_names:
        values = bootstrap_frame[name].to_numpy(dtype=np.float64)
        valid = values[np.isfinite(values)]
        ci[name] = {
            "lower_2_5": float(np.quantile(valid, 0.025, method="linear")) if len(valid) else None,
            "upper_97_5": float(np.quantile(valid, 0.975, method="linear")) if len(valid) else None,
            "valid_replicates": int(len(valid)),
            "invalid_replicates": int(len(values) - len(valid)),
        }
    bootstrap_csv = bootstrap_frame.to_csv(index=False, float_format="%.17g")
    metrics_payload = {
        "schema_version": 1,
        "run_id": lr04_run_manifest["run_id"],
        "evaluation_status": "complete_rq1_evaluation_experimental_feature_v0_2" if all(item["valid_replicates"] >= 2 for item in ci.values()) else "partial_ci",
        "feature_version": lr04_run_manifest["feature_version"],
        "split_version": lr04_run_manifest["split_version"],
        "preprocessing_version": lr04_run_manifest["preprocessing_version"],
        "model_protocol_version": lr04_run_manifest["model_protocol_version"],
        "partition": "test",
        "rows": len(lr04_y),
        "threshold": lr04_threshold,
        "prediction_file": lr04_prediction_path.name,
        "prediction_sha256": lr04_prediction_sha,
        "model_sha256": lr04_run_manifest["test_inference"]["model_sha256"],
        "bootstrap_indices_file": lr04_indices_path.name,
        "bootstrap_indices_sha256": lr04_indices_manifest["indices_sha256"],
        "bootstrap_replicates_file": lr04_bootstrap_path.name,
        "bootstrap_replicates_sha256": hashlib.sha256(bootstrap_csv.encode("utf-8")).hexdigest(),
        "counts": counts,
        "point_metrics": point,
        "confidence_intervals_95_percentile": ci,
        "calibration_curve": curve,
        "computational_metrics_seconds": {
            "full_train_fit": lr04_run_manifest["fit_seconds"],
            "test_inference_including_preprocessing": lr04_run_manifest["test_inference"]["inference_seconds_including_preprocessing"],
        },
        "methods": {
            "roc_auc_approved": "sklearn roc_auc_score(y_approved, p_approved)",
            "average_precision_denied": "sklearn average_precision_score(1-y_approved, 1-p_approved)",
            "predicted_approved": "p_approved >= locked threshold",
            "brier_approved": "mean((p_approved-y_approved)^2)",
            "calibration_curve": "10 quantile bins from Test p_approved; duplicate edges collapsed; no bootstrap CI",
            "calibration_intercept": "logistic offset recalibration of y_approved on logit(clip(p,1e-15,1-1e-15)); slope fixed at 1",
            "calibration_slope": "unpenalized logistic recalibration of y_approved on logit(clip(p,1e-15,1-1e-15)); free intercept and slope",
            "bootstrap": "1000 shared stratified true-class row-index replicates, seed 42, model and threshold fixed; 2.5 and 97.5 empirical percentiles",
            "invalid_replicate": "record reason in invalid_metrics; exclude only the undefined scalar metric from its CI; do not resample",
            "timing_ci": "no prediction bootstrap CI for fit or inference time",
        },
    }
    metrics_json = json.dumps(metrics_payload, ensure_ascii=False, indent=2) + "\n"
    lr04_bootstrap_path.write_text(bootstrap_csv, encoding="utf-8", newline="")
    lr04_metrics_path.write_text(metrics_json, encoding="utf-8", newline="")
    lr04_run_manifest["rq1_test_evaluation"] = {
        "status": metrics_payload["evaluation_status"],
        "metrics_file": lr04_metrics_path.name,
        "metrics_sha256": hashlib.sha256(lr04_metrics_path.read_bytes()).hexdigest(),
        "bootstrap_metrics_file": lr04_bootstrap_path.name,
        "bootstrap_metrics_sha256": hashlib.sha256(lr04_bootstrap_path.read_bytes()).hexdigest(),
        "bootstrap_indices_file": lr04_indices_path.name,
        "bootstrap_indices_sha256": lr04_indices_manifest["indices_sha256"],
    }
    lr04_manifest_path.write_text(json.dumps(lr04_run_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8", newline="")
    invalid_total = sum(item["invalid_replicates"] for item in ci.values())
    print(f"Saved LR-04 metrics and 95% CIs. Invalid metric-replicate values: {invalid_total}.")


Existing LR-04 metrics match; no recomputation.


## LR-04 — Quy ước kỹ thuật đã dùng

**Phạm vi run.** Đây là đánh giá RQ1 của Logistic Regression baseline lr_baseline_v1 với feature lock v0.2 ở trạng thái experimental. Split là split_v1, preprocessing là dataset_b_preprocessing_v2, đầu vào gồm 12 raw features và pipeline LR đã fit trên toàn Train tạo 66 cột sau mã hóa. Đây không phải FINAL_MODEL_FEATURES của toàn dự án.

**Ranh giới dữ liệu và dự báo.** Train dùng để fit; Validation dùng để chọn ngưỡng; Test gồm 43.996 hồ sơ chỉ được đưa qua fitted pipeline đúng một lượt predict_proba. Không fit lại, không đổi feature, model hoặc ngưỡng theo Test. Dự báo được lưu theo record_id. Nhãn y_approved=1 là Approved, y_approved=0 là Denied; p_approved là P(Approved) và p_denied=1-p_approved. Ngưỡng đã khóa là 0,747: p_approved >= 0,747 dự báo Approved, còn lại là Denied. LR-03 chọn ngưỡng từ 0,001 đến 0,999 (bước 0,001) để tối đa Balanced Accuracy trên Validation; nếu hòa, lấy gần 0,5 nhất rồi lấy ngưỡng thấp hơn.

**Định nghĩa metric.** ROC-AUC tính trên y_approved và p_approved. Chỉ số PR cho Denied là average precision trên 1-y_approved và 1-p_approved (không phải diện tích hình thang). Precision, recall và F1 của Denied dùng nhãn dự báo tại ngưỡng đã khóa; Recall Approved dùng lớp Approved. Balanced Accuracy là trung bình Recall Approved và Recall Denied. Brier là trung bình bình phương (p_approved-y_approved).

**Calibration.** Curve dùng 10 bin theo quantile của p_approved trên Test; cạnh bin trùng được gộp, mỗi bin lưu số hồ sơ, xác suất trung bình và tỷ lệ Approved thực tế. Trước khi lấy logit, xác suất được clip vào [1e-15, 1-1e-15]. Calibration intercept được fit bằng logistic offset với slope cố định bằng 1. Calibration slope được fit bằng logistic recalibration không regularization với intercept tự do.

**Độ bất định.** Tạo một ma trận 1.000 bộ Test row indices dùng chung ở cấp Dataset B cho LR/RF/XGBoost. Trong mỗi replicate, lấy mẫu có hoàn lại riêng trong hai lớp thật với đúng cỡ lớp gốc; dùng NumPy Generator PCG64, seed 42. Giữ nguyên model và ngưỡng. CI 95% cho từng scalar predictive metric là percentile 2,5% và 97,5% theo NumPy quantile method=linear. Nếu một metric không xác định ở replicate nào, ghi lý do và loại riêng giá trị đó khỏi CI của metric ấy; không lấy mẫu bù. Calibration curve, training time và inference time không có CI từ bootstrap prediction này. So sánh model theo cặp về sau phải dùng chính các replicate IDs và indices đã lưu.

**Thời gian và truy vết.** Training time lấy từ lần full-Train fit trong run manifest. Test inference time đo một lần gọi predict_proba, bao gồm preprocessing trong pipeline, không gồm đọc/ghi file. Run manifest liên kết hash của model, threshold, prediction và bootstrap indices. Xem các artifact tương ứng: ../../artifacts/dataset_b/lr_baseline_v1/run_manifest.json, ../../artifacts/dataset_b/lr_baseline_v1/rq1_test_metrics.json và ../../artifacts/dataset_b/rq1_test_bootstrap_manifest.json.


## LR v1 — Bảng kết quả để đối chiếu

Bản chụp tĩnh từ artifact của lr_baseline_v1; nếu artifact được cập nhật, cần tạo lại bảng này. Đây là kết quả của experimental feature lock v0.2, chưa phải kết luận so sánh với RF/XGBoost hay FINAL_MODEL_FEATURES.

### Train CV và chọn ngưỡng

| Giai đoạn | Chỉ số | Giá trị |
| --- | --- | ---: |
| Train, 5-fold CV | Mean ROC-AUC | 0.807934 |
| Train, 5-fold CV | Std ROC-AUC (ddof=0) | 0.002015 |
| Validation | Threshold đã khóa | 0.747 |
| Validation | Balanced Accuracy tại ngưỡng đã chọn | 0.739151 |

### Test: point metrics và 95% CI

Test có 43,996 hồ sơ. CI dùng 1.000 stratified bootstrap replicates; mỗi metric có 1.000 giá trị hợp lệ và 0 giá trị bị loại.

| Nhóm | Metric | Point | CI 95% [2,5%; 97,5%] |
| --- | --- | ---: | ---: |
| Discrimination | ROC-AUC Approved | 0.809249 | [0.804004; 0.814113] |
| Denied | Average precision Denied | 0.691383 | [0.684294; 0.698715] |
| Denied | Precision Denied | 0.583492 | [0.576395; 0.590955] |
| Denied | Recall Denied | 0.629103 | [0.620246; 0.637870] |
| Denied | F1 Denied | 0.605440 | [0.598376; 0.612532] |
| Classification | Recall Approved | 0.846991 | [0.842998; 0.851135] |
| Classification | Balanced Accuracy | 0.738047 | [0.733079; 0.743092] |
| Probability quality | Brier score | 0.128880 | [0.127267; 0.130465] |
| Calibration | Calibration intercept | -0.015376 | [-0.030905; -0.001345] |
| Calibration | Calibration slope | 1.021217 | [0.997895; 1.046998] |

### Ma trận phân lớp và thời gian

| Nhãn thật | Dự báo Approved | Dự báo Denied | Tổng |
| --- | ---: | ---: | ---: |
| Approved | 27,794 | 5,021 | 32,815 |
| Denied | 4,147 | 7,034 | 11,181 |

- Full-Train fit: 4.977 giây.
- Test inference, bao gồm preprocessing: 0.496 giây.
- Không tính CI bootstrap cho thời gian.

### Calibration curve trên Test

| Bin quantile | Khoảng p_approved | Số hồ sơ | P(Approved) trung bình | Tỷ lệ Approved thực tế |
| ---: | --- | ---: | ---: | ---: |
| 1 | [0.000000; 0.344068] | 4,400 | 0.106729 | 0.100909 |
| 2 | [0.344068; 0.665979] | 4,399 | 0.545633 | 0.548761 |
| 3 | [0.665979; 0.754647] | 4,400 | 0.721059 | 0.701591 |
| 4 | [0.754647; 0.795632] | 4,399 | 0.776464 | 0.782678 |
| 5 | [0.795632; 0.833330] | 4,400 | 0.814715 | 0.797500 |
| 6 | [0.833330; 0.882123] | 4,399 | 0.855188 | 0.847238 |
| 7 | [0.882123; 0.904769] | 4,400 | 0.896416 | 0.902500 |
| 8 | [0.904769; 0.914733] | 4,399 | 0.910900 | 0.934985 |
| 9 | [0.914733; 0.921453] | 4,400 | 0.918334 | 0.924773 |
| 10 | [0.921453; 0.993697] | 4,400 | 0.933139 | 0.917727 |

Nguồn đầy đủ: ../../artifacts/dataset_b/lr_baseline_v1/rq1_test_metrics.json; replicate-level values: ../../artifacts/dataset_b/lr_baseline_v1/rq1_bootstrap_metrics.csv. Giá trị hiển thị được làm tròn sáu chữ số thập phân; artifact giữ độ chính xác gốc.
